In [2]:
import torch
import torch.nn as nn
import math

# Vanilla Attention

In [3]:
class attn(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.Wq = nn.Linear(hidden_dim, hidden_dim)
        self.Wk = nn.Linear(hidden_dim, hidden_dim)
        self.Wv = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        scores = Q @ K.transpose(-2, -1)
        scores = scores / math.sqrt(Q.shape[-1])
        wts = torch.softmax(scores, dim = -1)
        output = wts @ V
        return output

In [4]:
model = attn(hidden_dim = 128)
x = torch.randn(4, 20, 128)
model(x)

tensor([[[-0.2343,  0.1036,  0.0650,  ..., -0.0011, -0.1765, -0.0464],
         [-0.1767,  0.1316,  0.0795,  ..., -0.0195, -0.1977, -0.0037],
         [-0.2253,  0.2199,  0.1150,  ..., -0.1114, -0.3346, -0.0184],
         ...,
         [-0.2430,  0.1996,  0.0237,  ..., -0.0802, -0.2896, -0.0951],
         [-0.1397,  0.1398,  0.0484,  ..., -0.0325, -0.2535, -0.0497],
         [-0.2230,  0.0424,  0.0225,  ..., -0.0453, -0.1710, -0.0780]],

        [[-0.0159,  0.0343,  0.0411,  ...,  0.1255, -0.0852, -0.1144],
         [-0.1050,  0.0540,  0.0620,  ...,  0.0766, -0.0192, -0.0320],
         [-0.0876,  0.0815,  0.0360,  ...,  0.1190, -0.0403, -0.0722],
         ...,
         [-0.0602,  0.0227,  0.0700,  ...,  0.0894, -0.0557, -0.1295],
         [-0.0495,  0.0438,  0.0975,  ...,  0.0462, -0.0784, -0.0754],
         [-0.1106,  0.0969, -0.0026,  ...,  0.0796, -0.0591, -0.0399]],

        [[-0.0240, -0.0809,  0.1398,  ..., -0.0545, -0.0938,  0.0678],
         [-0.0458, -0.1651,  0.1726,  ..., -0

In [6]:
model(x).shape

torch.Size([4, 20, 128])

# Multihead Attention

In [45]:
class MHA(nn.Module):
    def __init__(self, hidden_dim, n_heads):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_heads = n_heads

        self.head_dim = hidden_dim // n_heads

        self.Wq = nn.Linear(hidden_dim, hidden_dim)
        self.Wk = nn.Linear(hidden_dim, hidden_dim)
        self.Wv = nn.Linear(hidden_dim, hidden_dim)
        self.Wo = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        batch_size, seq_len, hidden_dim = x.shape
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        Q = Q.view(batch_size, seq_len, self.n_heads, -1).transpose(1,2)
        K = K.view(batch_size, seq_len, self.n_heads, -1).transpose(1,2)
        V = V.view(batch_size, seq_len, self.n_heads, -1).transpose(1,2)

        scores = Q @ K.transpose(-2, -1)
        scores = scores // math.sqrt(self.head_dim)

        weights = torch.softmax(scores, dim = -1)
        output = weights @ V
        output = output.transpose(1, 2).reshape(batch_size, seq_len, hidden_dim)
        output = self.Wo(output)

        return output

In [46]:
model = MHA(128, 8)

In [47]:
model(x)

tensor([[[-0.1122, -0.1257, -0.0858,  ...,  0.0813,  0.0308, -0.1042],
         [-0.2410, -0.1519, -0.1067,  ...,  0.0204, -0.0451, -0.0561],
         [-0.2483, -0.2027, -0.0583,  ...,  0.0619, -0.0429, -0.0579],
         ...,
         [-0.1447, -0.1725, -0.0583,  ..., -0.0582,  0.0755, -0.1286],
         [-0.1796, -0.1193, -0.1336,  ...,  0.0976,  0.0109, -0.1465],
         [-0.1905, -0.1406, -0.0506,  ...,  0.0200,  0.0038, -0.1068]],

        [[ 0.1283,  0.0988,  0.0191,  ...,  0.0413, -0.0166, -0.1826],
         [ 0.1044,  0.0903,  0.0200,  ...,  0.0071, -0.0204, -0.1684],
         [ 0.1356,  0.0697,  0.0597,  ...,  0.0532,  0.0119, -0.1234],
         ...,
         [ 0.0631,  0.0251,  0.0364,  ...,  0.0362, -0.0132, -0.1091],
         [ 0.1194,  0.0322, -0.0078,  ...,  0.0368, -0.0420, -0.1095],
         [ 0.1374,  0.0184, -0.0067,  ..., -0.0168,  0.0286, -0.0738]],

        [[ 0.0413,  0.0169, -0.1106,  ..., -0.0515,  0.0447, -0.1243],
         [ 0.0610,  0.1140, -0.0385,  ..., -0

# Masked (Causal) Attention

In [101]:
class masked(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.Wq = nn.Linear(hidden_dim, hidden_dim)
        self.Wk = nn.Linear(hidden_dim, hidden_dim)
        self.Wv = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        scores = Q @ K.transpose(-2,-1)
        mask = torch.triu(torch.ones(x.shape[-2], x.shape[-2]), diagonal = 1).bool()
        scores = scores.masked_fill(mask, float('-inf'))
        print(scores[0])
        scores = scores / math.sqrt(K.shape[-1])
        wts = torch.softmax(scores, dim = -1)
        output = wts @ V
        return output

In [102]:
model = masked(128)

In [103]:
x = torch.randn(4, 20, 128)
model(x)

tensor([[-3.5568,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf,    -inf,    -inf],
        [ 1.7813, -2.6097,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf,    -inf,    -inf],
        [-2.5669,  5.5335,  0.0819,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf,    -inf,    -inf],
        [ 5.8032, -3.5186,  7.3677,  2.0926,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf,    -inf,    -inf],
        [ 5.3786,  0.0130, -0.9171,  1.2968,  7.7211,    -inf,    -inf,    -inf,
            -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
      

tensor([[[ 0.3470, -0.9577,  0.6502,  ...,  0.0318, -0.6788,  0.6545],
         [ 0.1729, -0.6037,  0.2090,  ..., -0.0108, -0.1762,  0.7040],
         [ 0.0345, -0.6018, -0.1391,  ...,  0.1289,  0.4720,  0.6470],
         ...,
         [-0.0110,  0.2673,  0.0048,  ...,  0.0761,  0.3258, -0.1654],
         [ 0.0512,  0.2314,  0.0925,  ..., -0.0228,  0.2652, -0.0651],
         [ 0.1713,  0.2342, -0.0745,  ..., -0.0557,  0.3344, -0.1561]],

        [[-0.0227, -0.0594, -0.8209,  ...,  0.3533, -1.5147,  0.4605],
         [ 0.3762, -0.3254, -0.1979,  ...,  0.3072, -0.9260,  0.2938],
         [ 0.1228, -0.2271, -0.2353,  ...,  0.0289, -0.3620,  0.3608],
         ...,
         [ 0.2022, -0.0717, -0.2591,  ...,  0.0697, -0.1784,  0.2404],
         [ 0.1961, -0.0291, -0.2069,  ...,  0.0560, -0.0822,  0.2533],
         [ 0.2398, -0.0172, -0.2158,  ...,  0.0265, -0.2294,  0.2146]],

        [[-0.3570,  0.4304, -0.4036,  ..., -0.7127, -0.0742,  0.3482],
         [-0.1838,  0.4625, -0.5273,  ..., -0

# Cross Attention

In [104]:
class cross(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.Wq = nn.Linear(hidden_dim, hidden_dim)
        self.Wk = nn.Linear(hidden_dim, hidden_dim)
        self.Wv = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, enc_x, dec_x):
        Q = self.Wq(dec_x)
        K = self.Wk(enc_x)
        V = self.Wv(enc_x)

        scores = Q @ K.transpose(-2,-1)
        scores = scores / math.sqrt(K.shape[-1])
        wts = torch.softmax(scores, dim = -1)
        output = wts @ V
        return output

In [105]:
model = cross(128)
enc_x = torch.randn([4, 20, 128])
dec_x = torch.randn([4, 8, 128])
model(enc_x, dec_x)

tensor([[[-0.1613, -0.1090, -0.0926,  ...,  0.3264,  0.1731, -0.0339],
         [-0.2662, -0.0046, -0.2055,  ...,  0.3010,  0.0441, -0.1511],
         [-0.2040, -0.1417, -0.1055,  ...,  0.2190,  0.0735, -0.0927],
         ...,
         [-0.2013, -0.0608, -0.1572,  ...,  0.2925,  0.1100, -0.0788],
         [-0.2707, -0.0938, -0.1996,  ...,  0.2924,  0.0836, -0.1168],
         [-0.2196,  0.0621, -0.2251,  ...,  0.3786,  0.1506, -0.0385]],

        [[-0.1332, -0.0819, -0.1744,  ...,  0.2360, -0.0418, -0.0139],
         [-0.0633, -0.0729, -0.1248,  ...,  0.1790, -0.0908,  0.0722],
         [-0.0555, -0.0962, -0.1727,  ...,  0.1958, -0.2088,  0.0765],
         ...,
         [-0.1096, -0.0975, -0.1721,  ...,  0.2072, -0.0957,  0.0313],
         [-0.0607, -0.0981, -0.0043,  ...,  0.1446, -0.0683,  0.1655],
         [-0.1326, -0.0384, -0.0677,  ...,  0.1819, -0.0157,  0.0961]],

        [[-0.0520, -0.0702,  0.2534,  ...,  0.0826,  0.0234, -0.0504],
         [-0.0851, -0.0840,  0.2639,  ...,  0

# Transformer

In [125]:
class transformer(nn.Module):
    def __init__(self, n_heads, hidden_dim, nn_dim):
        super().__init__()
        self.n_heads = n_heads
        self.hidden_dim = hidden_dim
        self.nn_dim = nn_dim

        self.Wq = nn.Linear(hidden_dim, hidden_dim)
        self.Wk = nn.Linear(hidden_dim, hidden_dim)
        self.Wv = nn.Linear(hidden_dim, hidden_dim)
        self.Wo = nn.Linear(hidden_dim, hidden_dim)

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

        self.nn1 = nn.Linear(hidden_dim, nn_dim)
        self.relu = nn.ReLU()
        self.nn2 = nn.Linear(nn_dim, hidden_dim)

        self.head_dim = int(self.hidden_dim / self.n_heads)

    def forward(self, x):
        batch_size, seq_len, hidden_dim = x.shape

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        Q = Q.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1,2)
        K = K.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1,2)
        V = V.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1,2)
        
        
        scores = Q @ K.transpose(-2,-1)
        scores = scores / math.sqrt(K.shape[-1])
        weights = torch.softmax(scores, dim = -1)
        output = weights @ V
        output = output.transpose(1,2).reshape(batch_size, seq_len, hidden_dim)
        output = self.Wo(output)

        output = output + x
        output = self.norm1(output)
        residual = output
        output = self.nn1(output)
        output = self.relu(output)
        output = self.nn2(output)
        output = output + residual
        output = self.norm2(output)
        
        return output

In [126]:
model = transformer(8, 128, 256)
x = torch.randn(4,20,128)
model(x)

tensor([[[ 2.4388e+00,  6.1399e-01,  2.8163e+00,  ...,  7.5381e-01,
          -9.4472e-01, -4.5124e-01],
         [ 1.0372e+00, -2.1570e-01, -6.5988e-01,  ..., -1.1275e+00,
          -1.2039e+00,  4.5660e-01],
         [-6.5433e-01,  8.8046e-02,  1.6985e+00,  ...,  1.1977e-02,
          -3.0163e-01,  3.1510e-01],
         ...,
         [ 2.8265e-01,  3.0712e-01, -1.2072e+00,  ...,  8.8960e-01,
           3.3353e-01,  4.8277e-01],
         [-1.2701e+00, -2.0913e+00,  8.1631e-01,  ..., -9.7859e-01,
           6.2914e-01,  5.7210e-01],
         [ 1.6427e-01,  1.3911e+00, -1.0676e+00,  ..., -5.2169e-01,
          -1.6209e+00,  2.2686e+00]],

        [[-7.3637e-01, -7.8342e-01, -5.9764e-03,  ..., -6.9584e-01,
          -9.3592e-01, -3.8436e-01],
         [-1.3550e+00,  6.9138e-01,  1.4251e+00,  ..., -1.7800e+00,
          -1.5282e+00,  1.2478e-01],
         [-8.1431e-01,  4.8069e-01,  8.2044e-01,  ...,  1.4930e+00,
           1.1428e+00, -2.0220e-01],
         ...,
         [-1.6819e+00, -1

In [127]:
class blocks(nn.Module):
    def __init__(self, n_layers, n_heads, hidden_dim, nn_dim):
        super().__init__()
        self.blocks = nn.ModuleList([transformer(n_heads, hidden_dim, nn_dim) for _ in range(n_layers)])

    def forward(self, x):
        for block in self.blocks:
            x = block(x)

        return x

In [130]:
model = blocks(6, 8, 128, 256)
x = torch.randn(4, 20, 128)
model(x)

tensor([[[ 0.1635,  0.3517,  1.4718,  ..., -1.1104, -0.4134, -1.1116],
         [ 0.3112, -0.8670,  1.8105,  ...,  0.1807, -0.1051,  0.5568],
         [-0.1278, -0.4001,  0.1799,  ..., -0.0431,  0.7878,  0.3103],
         ...,
         [-1.2954, -0.8935,  1.5254,  ..., -0.1200, -0.2426,  0.9504],
         [ 0.8075, -1.1464, -0.1906,  ..., -0.9780, -0.5871,  0.5054],
         [ 1.3993,  0.4952,  2.0864,  ..., -0.3223, -0.9082,  1.6067]],

        [[ 0.4647,  0.6988,  1.7421,  ..., -1.8150, -0.0034,  1.9652],
         [ 1.6003,  0.1383,  0.7626,  ..., -1.1400, -0.6425, -0.1369],
         [-1.1565, -0.4610, -0.0586,  ..., -0.6303, -0.6847, -0.7350],
         ...,
         [-0.0114,  0.5299, -0.8079,  ..., -2.7751, -0.1098,  0.6227],
         [ 1.9946,  0.4587,  0.2147,  ...,  1.2554,  0.0048,  0.3815],
         [ 0.8387,  0.8147, -1.4373,  ...,  0.5220,  0.1205,  0.4478]],

        [[ 0.8935, -0.6783,  1.4624,  ..., -1.0382, -0.2956,  0.2227],
         [-0.0743, -0.5755,  2.3911,  ..., -1